# 章节练习

本练习覆盖 **实践与问题定位** 模块的核心知识点：GE 对接 PyTorch（TorchAir 图模式，5.2）、GE 对接 TensorFlow（TF Adapter / Parser，5.3），以及编译失败与运行报错的常见问题定位方法（5.4）。

完成下列题目检验学习效果，如有错误，建议结合对应小节复盘。题目分四层：判断题、单选题、多选题、实践题（排障演练）。

## 一、判断题

1. （判断题）TorchAir 是 Ascend Extension for PyTorch（torch_npu）中支持图模式能力的扩展库，负责把 PyTorch 的 FX 图（aten IR）转换为 AscendIR 计算图，再交给 GE 编译执行。

2. （判断题）使用 `torch.compile(model, backend=npu_backend)` 包装模型后，会在定义 `opt_model` 的那一刻立即完成编译。

3. （判断题）`aclgrphParseTensorFlow` 用于将 TensorFlow 的 `.pb` 模型解析为 AscendIR `Graph`，返回值 `GRAPH_SUCCESS(0)` 表示成功。

4. （判断题）使用 `aclgrphParseTensorFlow` 解析得到的 Graph 名固定不变，多次调用不会变化。

5. （判断题）TorchAir、TF Parser、GE 手工构图三种方式最终都收敛为同一种 AscendIR 图，进入统一的 GE 编译执行流程。

6. （判断题）`export ASCEND_GLOBAL_LOG_LEVEL=1` 表示把 CANN 日志级别设为 INFO，比默认更详细，常用于排障。

7. （判断题）错误码 `E19999`（形如 E*9***）一般表示用户类配置错误，按手册即可自助解决，无需联系技术支持。

8. （判断题）`DUMP_GRAPH_LEVEL=3` 表示 dump 出最后的生成图（经过 GE 优化、编译后的图）。

## 二、单选题

9. （单选题）TorchAir 图模式的端到端链路，正确的顺序是？
    A. PyTorch forward → AscendIR → FX Graph → GE 编译执行
    B. PyTorch forward → Dynamo/FX 图（aten IR）→ AscendIR → GE 编译执行
    C. PyTorch forward → OM → ATC → GE 编译执行
    D. PyTorch forward → ONNX → Parser → GE 编译执行

10. （单选题）若使用 ATC CLI 将 TensorFlow `.pb` 模型离线编译为 OM，正确的路径是？
    A. TF 脚本 → TorchAir → GE
    B. model.pb → ATC（`--framework=3`）→ OM → ACL
    C. model.pb → torch.compile → OM
    D. TF 脚本 → ONNX → aclgrphParseTensorFlow

> 若使用 `aclgrphParseTensorFlow`，应在程序内将得到的 `ge::Graph` 交给 GE Build API；Parser API 与 ATC CLI 是两条替代路线。

11. （单选题）三种图 dump 格式中，「信息最完整、可转 JSON」的是哪种？
    A. readable（`ge_readable*.txt`）
    B. onnx（`ge_onnx*.pbtxt`）
    C. ge_proto（`ge_proto*.txt`）
    D. plog

12. （单选题）执行时报 `Output Size mismatch ... model expect ..., but given ...`，最合理的处理是？
    A. 降低日志级别
    B. 按报错提示的 Expected size 重新分配输入/输出 buffer，并核对输入 shape/dtype
    C. 重新安装 driver
    D. 关闭图 dump

13. （单选题）日志中出现 `IR for op[%s] optype[%s] is not registered.`，最可能的原因是？
    A. 输出 buffer 太小
    B. 算子原型 so 未加载成功，或原型未编译进 so（可用 nm -D 查符号表）
    C. stream 同步超时
    D. soc_version 不匹配

14. （单选题）某模型每次推理输入 shape 都不同且未开启动态 shape，使用 TorchAir 图模式最可能出现的问题是？
    A. 结果数值错误
    B. 频繁 recompile，性能下降
    C. 无法导入 torchair
    D. 模型无法放到 NPU 设备

15. （单选题）以下关于 `aclgrphParseTensorFlow` 的 `parser_params` 描述，哪个正确？
    A. key/value 必须是 int 类型
    B. key 为参数类型、value 为参数值，均为 `AscendString` 格式
    C. 只能传入一个参数
    D. 该参数是输出参数

## 三、多选题

16. （多选题）以下哪些情况可能触发 graph break 或回退 Eager？
    A. 依赖张量值的 Python `if`（如 `if x.item() > 0`）
    B. backend 不支持的算子或自定义 Python 逻辑
    C. 调用 `.item()` / `.numpy()` 导致同步、跳出图
    D. 输入数据依赖的动态 shape 且未声明动态

17. （多选题）以下哪些是 TensorFlow Parser 支持的配置参数？
    A. `INPUT_SHAPE`（指定输入 shape，支持静态/范围/标量）
    B. `OUT_NODES`（指定输出节点）
    C. `OUTPUT`（指定转图后计算图名称）
    D. `ENABLE_SCOPE_FUSION_PASSES`（指定生效的 scope 融合规则，仅 TF Parser 支持）

18. （多选题）以下工具与适用场景的对应关系，哪些正确？
    A. msaicerr：分析 AI Core Error 问题，输出 info.txt 报告
    B. asys：一键式故障信息收集，业务卡住时可实时堆栈导出
    C. msprof：性能分析，采集算子级耗时 / FP-BP / AI Core metrics
    D. Netron：打开 `ge_onnx*.pbtxt` 可视化查看图结构与算子属性

19. （多选题）内存 OOM 定位时，以下做法正确的有？
    A. 在 plog 中搜索 `mem_stats` 查看各组件内存统计
    B. `aclrtMallocHost` 报错通常指向 Host 内存 OOM
    C. `aclrtMalloc` / `aclrtMallocPhysical` 报错通常指向 Device 内存 OOM
    D. 搜索 `DEV_PROC_MEM` 可查看 Device 业务进程内存统计

20. （多选题）以下关于 TorchAir 接入的最佳实践，哪些正确？
    A. 导入顺序应为 torch → torch_npu → torchair
    B. 模型与输入都需放到 NPU 设备（`.npu()`）
    C. 应先验证「跑通」（与 Eager 结果一致），再优化性能
    D. shape 多变场景应显式开启动态 shape，减少重复编译

## 四、实践题（排障演练）

> 实践题不设唯一答案，请按照课程步骤完成实践，记录运行现象，并说明对应的排查入口。可结合 answer 中的参考思路自评。

21. （TorchAir 实践）选一个小模型（如一个两层 MLP），用 `torch.compile(model, backend=npu_backend)` 跑成 TorchAir 图模式。请：
    - 写出导入顺序与关键代码（模型/输入需 `.npu()`）；
    - 先验证图模式输出与 Eager 输出一致（跑通）；
    - 记录「首次调用」与「后续调用」的耗时差异，并解释原因。

22. （TensorFlow 离线编译实践）选择一种独立的 TensorFlow `.pb` 离线编译路线，**不要把 Parser API 和 ATC CLI 串成一条调用链**：
    - 编程式路线：`aclgrphParseTensorFlow` → `aclgrphBuildInitialize` → `aclgrphBuildModel` → `aclgrphSaveModel` → `.om`；
    - 命令行路线：`atc --model=model.pb --framework=3 --output=model --soc_version=<soc>` → `model.om`。
    请记录你选择的路线、配置或命令、产物与关键成功日志。若选择 ATC，请使用 ATC 参数（如 `--input_shape` / `--out_nodes`）；Parser 的 `parser_params` 不会自动传给 ATC。

23. （排障演练）模型转换/加载时报 `IR for op ... is not registered.`、`have no ir factory`，或某算子被转成 `frameworkop` / 报 `it is not supported`。请按日志类型分别写出定位路径：
    - `frameworkop` / `it is not supported`：如何确认 TensorFlow 框架适配插件路径、加载日志与符号？
    - `IR ... is not registered` / `have no ir factory`：如何确认算子原型 so、`OpsProtoManager` 加载日志与符号？
    - 如何用 `nm -D` 检查对应 so？还应检查哪些环境变量（提示：算子包路径）？

24. （排障演练）离线推理执行时报 `Output Size mismatch ... model expect ..., but given ...`。请写出定位与修复步骤：
    - 如何确认是 buffer 大小问题而非数值问题？
    - 修复时按什么尺寸重新分配 buffer？
    - 动态 shape 场景还需额外核对什么？

25. （排障演练）一个 TorchAir 图模式推理服务时延偏高且抖动大，日志中频繁出现 recompile。请给出排查与优化思路：
    - 用什么手段确认是否在频繁重编译？
    - 若确认由 shape 多变导致，你会怎么处理（至少两项）？

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/05.05_answer.txt